# LCO‑HBI Analysis (Fixed Copy)
This simplified, cleaned notebook loads a solved PyPSA network, inspects `n.statistics()` output, and computes a defensible LCO‑HBI using a robust fallback if statistics are unavailable.

In [31]:
# Load and compare every solved network in a folder
import pathlib
from datetime import datetime
from typing import Any

import pandas as pd
import pypsa
import seaborn as sns
import plotly.graph_objects as go
from plotly.io import renderers
from IPython.display import display

renderers.default = "vscode"

print("Run at", datetime.now().isoformat() + "Z")
print("pypsa version", getattr(pypsa, "__version__", "unknown"))

repo_root = pathlib.Path("../..").resolve()
network_folder = pathlib.Path(r"resources/lco-hbi/cost_year~2050/Europe_reserved")
if not network_folder.is_absolute():
    network_folder = (repo_root / network_folder).resolve()

network_files = sorted(network_folder.rglob("*.nc"))
if not network_files:
    raise FileNotFoundError(f"No .nc files found under {network_folder}")

print(f"Found {len(network_files)} network files under {network_folder}")


def network_label(path: pathlib.Path, root: pathlib.Path) -> str:
    try:
        rel_path = path.relative_to(root)
    except ValueError:
        rel_path = path
    return rel_path.with_suffix("").as_posix()


def collapse_stats_table(value: Any) -> pd.Series:
    """Return a Series indexed by the specific technology name, e.g. renewable_pv."""

    def comp_name_from_tuple(tup):
        parts = [str(x) for x in tup]
        if len(parts) >= 2:
            return "_".join(parts[1:])
        return parts[0]

    if isinstance(value, pd.Series):
        series = pd.to_numeric(value, errors="coerce").fillna(0)
        if isinstance(series.index, pd.MultiIndex):
            names = [comp_name_from_tuple(t) for t in series.index.values]
            return pd.Series(series.values, index=names).groupby(level=0).sum()
        return series
    if isinstance(value, pd.DataFrame):
        numeric = value.select_dtypes(include="number")
        if numeric.empty:
            return pd.Series(dtype=float)
        row_sum = numeric.sum(axis=1)
        if isinstance(row_sum.index, pd.MultiIndex):
            names = [comp_name_from_tuple(t) for t in row_sum.index.values]
            return pd.Series(row_sum.values, index=names).groupby(level=0).sum()
        return row_sum
    return pd.Series({"value": float(value)})


def hbi_annual_output_tons(network: pypsa.Network):
    hbi_loads = network.loads.index[
        network.loads["bus"].astype(str).str.contains("hbi", case=False, na=False)
    ].tolist()
    if not hbi_loads:
        raise ValueError("No HBI load found in network.loads.")

    load_ts = network.loads_t.p.reindex(columns=hbi_loads, fill_value=0)
    hbi_hourly_tons = load_ts.sum(axis=1)

    if "objective" in getattr(network, "snapshot_weightings", pd.DataFrame()).columns:
        weights = network.snapshot_weightings["objective"].reindex(
            hbi_hourly_tons.index
        )
    else:
        weights = pd.Series(1.0, index=hbi_hourly_tons.index)
    weights = weights.fillna(1.0)

    annual_tons = float((hbi_hourly_tons * weights).sum())
    return annual_tons, hbi_loads


def lcox_breakdown_per_ton(network: pypsa.Network):
    annual_hbi_tons, hbi_loads = hbi_annual_output_tons(network)
    if annual_hbi_tons <= 0:
        raise ValueError("Annual HBI output must be positive.")

    cost_tables = {
        "capex": collapse_stats_table(network.stats.capex()),
        "opex": collapse_stats_table(network.stats.opex()),
        "fom": collapse_stats_table(network.stats.fom()),
    }

    records = []
    for category, series in cost_tables.items():
        for technology, value in series.items():
            if float(value) == 0.0:
                continue
            records.append(
                {
                    "category": category,
                    "technology": str(technology),
                    "value_per_t_hbi": float(value) / annual_hbi_tons,
                }
            )

    breakdown = pd.DataFrame(records)
    return breakdown, annual_hbi_tons, hbi_loads


sns.set_theme(style="whitegrid", context="talk", font_scale=0.9)

technology_names = {
    "renewable_pv": "Solar PV",
    "renewable_wind_onshore": "Onshore wind",
    "renewable_wind_offshore": "Offshore wind",
    "battery_elec": "Battery storage",
    "electrolysis": "Electrolyzer",
    "hydrogen": "Hydrogen storage",
    "direct_reduction_furnace": "Direct reduction furnace",
}

category_names = {
    "capex": "Capex",
    "opex": "Opex",
    "fom": "FOM",
}

technology_order_preference = [
    "renewable_pv",
    "renewable_wind_onshore",
    "renewable_wind_offshore",
    "battery_elec",
    "electrolysis",
    "hydrogen",
    "direct_reduction_furnace",
]

breakdown_frames = []
summary_records = []
skipped_records = []
for network_path in network_files:
    network_name = network_label(network_path, network_folder)
    network = pypsa.Network(str(network_path))
    try:
        breakdown, annual_hbi_tons, hbi_loads = lcox_breakdown_per_ton(network)
    except ValueError as exc:
        skipped_records.append(
            {
                "network": network_name,
                "file": str(network_path),
                "reason": str(exc),
            }
        )
        print(f"Skipping {network_name}: {exc}")
        continue

    if breakdown.empty:
        skipped_records.append(
            {
                "network": network_name,
                "file": str(network_path),
                "reason": "Empty cost breakdown",
            }
        )
        print(f"Skipping {network_name}: empty cost breakdown")
        continue

    demand_mt = annual_hbi_tons / 1e6
    breakdown = breakdown.assign(
        network=network_name,
        file=str(network_path),
        hbi_loads=", ".join(hbi_loads),
        hbi_tons_per_year=annual_hbi_tons,
        demand_mt=demand_mt,
    )
    breakdown_frames.append(breakdown)
    summary_records.append(
        {
            "network": network_name,
            "file": str(network_path),
            "hbi_loads": ", ".join(hbi_loads),
            "hbi_tons_per_year": annual_hbi_tons,
            "demand_mt": demand_mt,
            "total_lcox_per_t_hbi": float(breakdown["value_per_t_hbi"].sum()),
        }
    )

if not breakdown_frames:
    raise ValueError(
        f"No networks with HBI cost breakdowns found under {network_folder}"
    )

long_df = pd.concat(breakdown_frames, ignore_index=True)
summary_df = (
    pd.DataFrame(summary_records).sort_values("demand_mt").reset_index(drop=True)
)

technology_order = [
    tech for tech in technology_order_preference if tech in set(long_df["technology"])
]

plot_df = long_df.pivot_table(
    index="demand_mt",
    columns=["technology", "category"],
    values="value_per_t_hbi",
    aggfunc="sum",
    fill_value=0,
)

all_columns = pd.MultiIndex.from_product([technology_order, ["capex", "opex", "fom"]])
plot_df = plot_df.reindex(columns=all_columns, fill_value=0)
present_columns = [col for col in plot_df.columns if float(plot_df[col].sum()) > 0]
plot_df = plot_df[present_columns]

display(summary_df)
display(plot_df)

if skipped_records:
    display(pd.DataFrame(skipped_records))

base_palette = sns.color_palette("tab10", n_colors=max(len(technology_order), 1))
technology_colors = {tech: base_palette[i] for i, tech in enumerate(technology_order)}


def rgba_color(base_color, category):
    if category == "capex":
        alpha = 0.98
    elif category == "opex":
        alpha = 0.78
    else:
        alpha = 0.58
    r, g, b = [int(round(c * 255)) for c in base_color]
    return f"rgba({r}, {g}, {b}, {alpha})"


bar_widths = [x * 0.9 for x in plot_df.index.tolist()]

tech_blocks = [
    (tech, [col for col in plot_df.columns if col[0] == tech])
    for tech in technology_order
    if any(col[0] == tech for col in plot_df.columns)
]

fig = go.Figure()
seen_labels = set()
for technology, cols in tech_blocks:
    for category in ["capex", "opex", "fom"]:
        column = (technology, category)
        if column not in plot_df.columns:
            continue
        values = plot_df[column]
        if float(values.sum()) <= 0:
            continue
        tech_label = technology_names.get(
            technology, technology.replace("_", " ").title()
        )
        cat_label = category_names.get(category, category.upper())
        label = f"{tech_label} · {cat_label}"
        fig.add_trace(
            go.Bar(
                x=plot_df.index.tolist(),
                y=values.values.tolist(),
                width=bar_widths,
                name=label if label not in seen_labels else label,
                marker_color=rgba_color(technology_colors[technology], category),
                marker_line_color="white",
                marker_line_width=0.6,
                hovertemplate=(
                    f"Demand: %{{x:.2f}} Mt/year"
                    f"<br>{tech_label}"
                    f"<br>{cat_label}: %{{y:.2f}} €/t HBI"
                    "<extra></extra>"
                ),
            )
        )
        seen_labels.add(label)

region_name = network_folder.name.replace("_reserved", "").replace("_", " ")
fig.update_layout(
    barmode="stack",
    title=f"{region_name} HBI LCOX breakdown",
    xaxis_title="HBI demand [Mt/year]",
    yaxis_title="Levelized Cost of HBI [€/t]",
    template="plotly_white",
    legend_title="Technology / cost",
    width=max(1100, 170 * len(plot_df.index)),
    height=700,
    bargap=0.0,
    font=dict(size=14),
    hovermode="x unified",
)
fig.update_xaxes(
    type="log",
    tickmode="array",
    tickvals=plot_df.index.tolist(),
    ticktext=[f"{d:g}" for d in plot_df.index],
)
fig.update_yaxes(gridcolor="rgba(0,0,0,0.1)")

# interactive_html = (repo_root / "results" / "figures_general" / f"{network_folder.name}_hbi_lcox_interactive.html")
# interactive_html.parent.mkdir(parents=True, exist_ok=True)
# fig.write_html(str(interactive_html), include_plotlyjs="cdn")
# print(f"Interactive HTML written to {interactive_html}")
display(fig)

plot_df

INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)


Run at 2026-05-20T17:43:03.061869Z
pypsa version 1.1.2
Found 6 network files under C:\Users\JanLeopoldTautorus\Repos\shift\resources\lco-hbi\cost_year~2050\Europe_reserved


INFO:pypsa.network.io:Imported network 'LCOX-Europe-hbi-0.01' has buses, carriers, generators, links, loads, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'LCOX-Europe-hbi-0.1' has buses, carriers, generators, links, loads, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'LCOX-Europe-hbi-1.0' has buses, carriers, generators, links, loads, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'LCOX-Europe-hbi-10.0' has buses, carriers, generators, links, loads, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'LCOX-Europe-hbi-100.0' has buses, carriers, generators, links, loads, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.

,network,file,hbi_loads,hbi_tons_per_year,demand_mt,total_lcox_per_t_hbi
0,network_0.01,C:\Users\JanLeopoldTautorus\Repos\shift\resour...,hbi_demand,1.000000e+04,0.01,546.343745
1,network_0.1,C:\Users\JanLeopoldTautorus\Repos\shift\resour...,hbi_demand,1.000000e+05,0.10,546.343000
2,network_1,C:\Users\JanLeopoldTautorus\Repos\shift\resour...,hbi_demand,1.000000e+06,1.00,546.343006
3,network_10,C:\Users\JanLeopoldTautorus\Repos\shift\resour...,hbi_demand,1.000000e+07,10.00,546.343006
4,network_100,C:\Users\JanLeopoldTautorus\Repos\shift\resour...,hbi_demand,1.000000e+08,100.00,546.343292
5,network_1000,C:\Users\JanLeopoldTautorus\Repos\shift\resour...,hbi_demand,1.000000e+09,1000.00,562.528782


renewable_pv           renewable_wind_onshore             \
                 capex       fom                  capex        fom   
demand_mt                                                            
0.01         21.560359  5.332561             227.544311  25.257855   
0.10         21.560462  5.332587             227.543480  25.257763   
1.00         21.560461  5.332587             227.543486  25.257763   
10.00        21.560461  5.332587             227.543486  25.257763   
100.00       21.432149  5.300851             227.677715  25.272663   
1000.00      36.784441  9.097960             213.439743  23.692221   

          renewable_wind_offshore                battery_elec electrolysis  \
                            capex           fom         capex        capex   
demand_mt                                                                    
0.01                 1.880000e-07  3.800000e-08  9.000000e-09    41.477669   
0.10                 4.490000e-08  9.200000e-09  2.200000e-09    41.477673   
1.00                 2.432000e-08  4.960000e-09  1.210000e-09    41.477674   
10.00                8.000000e-12  2.000000e-12  0.000000e+00    41.477674   
100.00               0.000000e+00  0.000000e+00  0.000000e+00    41.449407   
1000.00              2.486100e-10  5.075000e-11  1.682000e-11    51.756010   

                      hydrogen direct_reduction_furnace              
                 fom     capex                    capex         fom  
demand_mt                                                            
0.01       11.357142  4.193414                99.580606  110.039829  
0.10       11.357144  4.193447                99.580610  110.039834  
1.00       11.357144  4.193447                99.580610  110.039834  
10.00      11.357144  4.193447                99.580610  110.039834  
100.00     11.349404  4.229800                99.585769  110.045535  
1000.00    14.171490  3.435460                99.832869  110.318588

renewable_pv           renewable_wind_onshore             \
                 capex       fom                  capex        fom   
demand_mt                                                            
0.01         21.560359  5.332561             227.544311  25.257855   
0.10         21.560462  5.332587             227.543480  25.257763   
1.00         21.560461  5.332587             227.543486  25.257763   
10.00        21.560461  5.332587             227.543486  25.257763   
100.00       21.432149  5.300851             227.677715  25.272663   
1000.00      36.784441  9.097960             213.439743  23.692221   

          renewable_wind_offshore                battery_elec electrolysis  \
                            capex           fom         capex        capex   
demand_mt                                                                    
0.01                 1.880000e-07  3.800000e-08  9.000000e-09    41.477669   
0.10                 4.490000e-08  9.200000e-09  2.200000e-09    41.477673   
1.00                 2.432000e-08  4.960000e-09  1.210000e-09    41.477674   
10.00                8.000000e-12  2.000000e-12  0.000000e+00    41.477674   
100.00               0.000000e+00  0.000000e+00  0.000000e+00    41.449407   
1000.00              2.486100e-10  5.075000e-11  1.682000e-11    51.756010   

                      hydrogen direct_reduction_furnace              
                 fom     capex                    capex         fom  
demand_mt                                                            
0.01       11.357142  4.193414                99.580606  110.039829  
0.10       11.357144  4.193447                99.580610  110.039834  
1.00       11.357144  4.193447                99.580610  110.039834  
10.00      11.357144  4.193447                99.580610  110.039834  
100.00     11.349404  4.229800                99.585769  110.045535  
1000.00    14.171490  3.435460                99.832869  110.318588